# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [5]:
import pandas as pd
df = pd.read_csv(r"C:\Users\priya\AIcoursework\week7\lab-chains-in-langchain\data\Data.csv")


In [6]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [ ]:
!pip install langchain langchain-core langchain-openai langchain-community langchain-classic

In [ ]:
!pip show langchain langchain-core langchain-openai langchain-community

In [7]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [8]:
#Replace None by your own value and justify
llm = ChatOpenAI(temperature=None)


In [9]:
prompt = ChatPromptTemplate.from_template( "Describe the product type: {product}.\n\n"
    "Include:\n"
    "1. The target customer\n"
    "2. Three key features\n"
    "3. One short marketing sentence\n"
    "Keep the answer concise and professional."
)

In [11]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

In [13]:
product = "noise-cancelling wireless headphones"
chain.invoke(product)

'1. The target customer for noise-cancelling wireless headphones would be individuals who value high-quality sound, portability, and convenience.\n\n2. Three key features of noise-cancelling wireless headphones include:\n   - Active noise cancellation technology for immersive listening experience\n   - Wireless connectivity for freedom of movement\n   - Long-lasting battery life for extended use\n\n3. Immerse yourself in crystal-clear sound with our noise-cancelling wireless headphones, perfect for busy commuters and music enthusiasts alike.'

## SimpleSequentialChain

In [14]:
from langchain_classic.chains import SimpleSequentialChain  

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Create a short product concept for the following product type: {product}.\n\n"
    "Mention the target audience, main value proposition, and two differentiating feature."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)



In [ ]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "You are a marketing copywriter.\n\n"
    "Using the product concept below, create:\n"
    "1. A catchy slogan\n"
    "2. A 2-sentence social media caption\n"
    "3. Three short hashtags\n\n"
    "Product concepts:\n{product_concept}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [17]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [19]:
overall_simple_chain.invoke(product)



> Entering new SimpleSequentialChain chain...
**Product Concept: SilentSphere Wireless Headphones**

**Target Audience:**  
Urban commuters, remote workers, and frequent travelers who seek to enhance their audio experience by eliminating background noise while enjoying music, podcasts, or taking calls.

**Main Value Proposition:**  
SilentSphere Wireless Headphones provide an immersive listening experience by employing advanced noise-cancellation technology, allowing users to enjoy their audio in any environment without distractions, fostering productivity and relaxation.

**Differentiating Features:**

1. **Adaptive Sound Control:**  
SilentSphere headphones utilize AI-driven adaptive sound control that automatically adjusts noise-cancellation levels based on the surrounding environment. Whether in a bustling coffee shop, on a train, or at home, users will experience optimal sound isolation tailored to their current setting.

2. **Wellness Mode:**  
Incorporating biofeedback technol

{'input': 'noise-cancelling wireless headphones',
 'output': '1. **Catchy Slogan:**  \n"Escape the Noise, Embrace the Sound."\n\n2. **Social Media Caption:**  \n"Transform your daily commute into a serene escape with SilentSphere Wireless Headphones. Experience the perfect blend of immersive sound and stress relief with our innovative adaptive noise cancellation and Wellness Mode. 🎧✨"\n\n3. **Short Hashtags:**  \n#SilentSphere #SoundYourWay #MindfulListening'}

**Repeat the above twice for different products**

## SequentialChain

In [20]:
from langchain_classic.chains import SequentialChain  

In [ ]:


first_prompt = ChatPromptTemplate.from_template(
    "Translate the following customer review into English.\n"
    "If it is already in English, clean the grammar lightly without changing the meaning.\n\n"
    "Review:\n{review}"
)

chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="english_review"
)

In [38]:
# Chain 2: summarize the translated review.
second_prompt = ChatPromptTemplate.from_template(
    "Summarize the following English customer review in one clear sentence.\n\n"
    "English review:\n{english_review}"
)

chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
)

In [39]:
# Chain 3: identify the original review language.
third_prompt = ChatPromptTemplate.from_template(
    "Identify the original language of the customer review.\n"
    "Return only the language name, for example: English, German, French, Spanish.\n\n"
    "Original review:\n{review}"
)

chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="language"
)



In [42]:
# Chain 4: create a follow-up message using outputs from previous chains.
fourth_prompt = ChatPromptTemplate.from_template(
    "You are a helpful customer support agent.\n\n"
    "Original language: {language}\n"
    "English review: {english_review}\n"
    "Summary: {summary}\n\n"
    "Write a polite follow-up message to the customer.\n"
    "Acknowledge the feedback, mention the main issue or praise, and offer a useful next step.\n"
    "Keep it professional and concise."
)

chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="followup_message"
)



In [43]:
# Overall chain
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["review"],
    output_variables=["english_review", "summary", "language", "followup_message"],
    verbose=True
)

In [44]:
review = df.Review[5]
overall_chain(review)

C:\Users\priya\AppData\Local\Temp\ipykernel_70200\1992003631.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  overall_chain(review)




> Entering new SequentialChain chain...

> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\r\nVieux lot ou contrefaçon !?",
 'english_review': "I find the taste mediocre. The foam doesn't hold, it's weird. I buy the same ones in stores and the taste is much better... \nOld batch or counterfeit!?",
 'summary': 'The reviewer found the taste mediocre and the foam unsatisfactory, suspecting that the product may be an old batch or counterfeit.',
 'language': 'French',
 'followup_message': 'Dear valued customer,\n\nThank you for sharing your feedback with us. We apologize for any disappointment you experienced with the taste and foam of our product. This is not the quality we strive to deliver to our customers.\n\nWe take your concerns seriously and would like to investigate further to ensure the quality of our products. Please reach out to our customer support team with more details about your purchase so we can assist you in resolving

In [60]:
for idx in [0, 3]:
    print("=" * 90)
    print(f"Review index: {idx}")
    result = overall_chain({"review": df.Review.iloc[idx]})
    print("Language:", result["language"])
    print("Summary:", result["summary"])
    print("Follow-up message:", result["followup_message"])

Review index: 0


> Entering new SequentialChain chain...

> Finished chain.
Language: English
Summary: The customer ordered a king-size set of sheets and had a minor issue with the pillowcases not matching, but customer service quickly resolved the issue and they were ultimately satisfied with the quality and price of the sheets.
Follow-up message: Dear Customer,

Thank you for sharing your feedback with us. We appreciate your input regarding the pillowcases not matching and are glad to hear that our customer service team was able to promptly assist you in resolving the issue.

We understand the importance of having matching pillowcases and apologize for any inconvenience this may have caused. If you have any further concerns or require any additional assistance, please do not hesitate to reach out to us.

We value your business and hope that you continue to enjoy the quality and coolness of your new sheets. Thank you for choosing our products.

Sincerely,
[Your Name]
Customer Support

**Repeat the above twice for different products or reviews**

## Router Chain

In [45]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [46]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [47]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  # Updated path for prompts (core package)

In [48]:
llm = ChatOpenAI(temperature=0)

In [49]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [50]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [51]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [52]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [53]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\priya\AppData\Local\Temp\ipykernel_70200\3038952769.py:1: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(router_chain=router_chain,


In [54]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [55]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [56]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for the development, functioning, and reproduction of all living organisms. DNA contains the information needed to build and maintain an organism, including the proteins that make up our cells and tissues. \n\nHaving DNA in every cell ensures that each cell has the necessary genetic information to carry out its specific functions and to replicate itself accurately during cell division. This ensures that the genetic information is passed on to the next generation of cells, maintaining the integrity and continuity of the organism's genetic code.\n\nAdditionally, DNA serves as a storage system for genetic information that can be accessed and utilized by cells as needed. This allows for the regulation of gene expression, the repair of damaged DNA, and the adaptation to changing environmental conditions.\n\nIn summary, every cell in our body contains DNA because it is essential for the pro

In [61]:
creative_questions = [
    "Explain recursion with a simple Python example.",
    "Why did the Industrial Revolution start in Britain?",
    "Plan a 3-day healthy meal plan for a busy student."
]

for question in creative_questions:
    print("=" * 90)
    print(f"Question: {question}")
    print(chain.run(question))

Question: Explain recursion with a simple Python example.


> Entering new MultiPromptChain chain...
computer science: {'input': 'Explain recursion with a simple Python example.'}
> Finished chain.
Recursion is a programming technique where a function calls itself in order to solve a problem. It is a powerful tool for solving problems that can be broken down into smaller, similar subproblems.

Here is a simple example of recursion in Python:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)

# Example usage
result = factorial(5)
print(result)  # Output: 120
```

In this example, the `factorial` function calculates the factorial of a given number `n`. If `n` is equal to 0, the function returns 1 (base case). Otherwise, it recursively calls itself with `n-1` until it reaches the base case. This process continues until `n` becomes 0, at which point the function starts returning the factorial values back up the call stack.

Recursion c

**Repeat the above at least once for different inputs and chains executions - Be creative!**